# Check 3+ with the paper's evaluation: walk-embedding baselines

Same edge groups and the same evaluation as `check3_paper_metrics.ipynb` (one sampled negative per positive with
`negative_samples`, AUROC, AUPRC, Recall@100 over all candidate tails), for the walk-embedding baselines of
`walk_coverage_baselines.ipynb`:
- `node2vec_classifier`, `node2vec_rel_classifier`, `dreamwalk_samecorpus`: trained on the same pseudo-sentences as NetMedGPT;
- `dreamwalk_full`: the complete DREAMwalk pipeline with its own walks (saved embeddings of `dreamwalk_full.py`).
As in `walk_coverage_baselines.ipynb`, the never-walked edges are removed from the classifiers' training pairs.
The positives and negatives are identical to those of the NetMedGPT evaluation (same edges, same sampling seed).

**Run with the `walk_baselines` kernel** (gensim, xgboost), after `check3_paper_metrics.ipynb`.
Output: `data/new_analysis/coverage/check3x_paper_metrics_baselines_per_seed.csv` and `check3x_paper_metrics_all_methods_summary.csv`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "revision_scripts":
    os.chdir("..")
import nbformat

# training code of walk_coverage_baselines.ipynb (path setup, settings, functions), unchanged
for cell in [c for c in nbformat.read("revision_scripts/walk_coverage_baselines.ipynb", as_version=4).cells if c.cell_type == "code"][:3]:
    exec(cell.source)

from sklearn.metrics import roc_auc_score, average_precision_score
from netmedgpt.sentence_generation import negative_samples

INFERENCES = ["random_link_split", "zero_shot_split"]
SEEDS = [1, 2, 3, 4, 5]
BASELINES = ["node2vec_classifier", "node2vec_rel_classifier", "dreamwalk_samecorpus", "dreamwalk_full"]
NODE_TYPES = {"indication": "disease|drug", "contraindication": "disease|drug", "off-label use": "disease|drug",
              "drug_protein": "drug|gene/protein", "drug_effect": "drug|effect/phenotype"}
GROUPS = {"seen": "training edge, shown in walks",
          "not adjacent, same walk": "training edge, missed by walks (nodes co-occur)",
          "not adjacent, never in same walk": "training edge, missed by walks (nodes never co-occur)",
          "test": "held-out test edge"}
COV_DIR = "data/new_analysis/coverage"

In [ ]:
# negative pool and Recall@100 candidates, identical to check3_paper_metrics.ipynb
nodes_kg = pd.read_csv("data/KG/nodes.csv")
edge_kg = pd.read_csv("data/KG/edges.csv")
rel_token = edge_kg.drop_duplicates("relation").set_index("relation")["z_index"].astype(int).to_dict()
tail_pool = {r: edge_kg.loc[(edge_kg["relation"] == r) & (edge_kg["node_types"] == nt), "y_index"].unique() for r, nt in NODE_TYPES.items()}
type_ids = {t: torch.tensor(nodes_kg.loc[nodes_kg["node_type"] == t, "node_index"].unique(), dtype=torch.int64)
            for t in ["drug", "gene/protein", "effect/phenotype"]}
candidates = {r: type_ids[nt.split("|")[1]] for r, nt in NODE_TYPES.items()}


def recall_at_k(pair_fn, z, pos, cand, k=100):
    """mean over heads of the share of their positives among the top-k candidates (as node_level_eval)"""
    fn = pair_score_matrix_fn(pair_fn, z, cand)
    by_head = pos.groupby("x_index")["y_index"].apply(set)
    heads = by_head.index.to_numpy()
    step = max(1, PAIR_BUDGET // len(cand))
    rec = []
    for i in range(0, len(heads), step):
        S = fn(heads[i:i + step])
        top = cand.numpy()[np.argsort(-S, axis=1)[:, :k]]
        rec += [len(by_head[h] & set(t)) / len(by_head[h]) for h, t in zip(heads[i:i + step], top)]
    return float(np.mean(rec))

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
rows = []
for inference in INFERENCES:
    for seed in SEEDS:
        print(f"===== {inference} | seed {seed} =====")
        edges = pd.read_csv(os.path.join(COV_DIR, f"{inference}_seed{seed}", "check3x_edges.csv"))
        edges = edges[(edges["method"] == "NetMedGPT") & edges["group"].isin(GROUPS)]

        split = load_split(inference, seed)
        data = build_corpus(split)
        g = build_graph(split)
        st = WalkStats(g)
        walks = data["node2vec_walks"].numpy()
        for i in range(0, len(walks), CHUNK):
            st.add(walks[i:i + CHUNK])
        scorers = build_baselines(split, g, data, never_walked(split, g, st), BASELINES, device, seed)

        for (rel, grp), e in edges.groupby(["relation", "group"]):
            pos = e.rename(columns={"head": "x_index", "tail": "y_index"})[["x_index", "y_index"]].drop_duplicates()
            neg = negative_samples(pos, edge_kg, tail_pool[rel], seed)
            pairs = np.vstack([pos.to_numpy(), np.array(neg, dtype=np.int64).reshape(-1, 2)])
            label = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
            z = rel_token[rel]
            for method, pair_fn in scorers.items():
                score = np.asarray(torch.as_tensor(pair_fn(torch.as_tensor(pairs[:, 0]), torch.as_tensor(pairs[:, 1]), z)).float().cpu())
                rows.append({"inference": inference, "seed": seed, "relation": rel, "group": GROUPS[grp], "method": method,
                             "positives": len(pos), "AUROC": roc_auc_score(label, score),
                             "AUPRC": average_precision_score(label, score),
                             "Recall@100": recall_at_k(pair_fn, z, pos, candidates[rel])})
        pd.DataFrame(rows).to_csv(os.path.join(COV_DIR, "check3x_paper_metrics_baselines_per_seed.csv"), index=False)
        del split, data, g, st, scorers

res = pd.DataFrame(rows)

In [ ]:
# ── NetMedGPT (check3_paper_metrics.ipynb) and baselines, mean over seeds ──────
nm = pd.read_csv(os.path.join(COV_DIR, "check3x_paper_metrics_per_seed.csv")).assign(method="NetMedGPT")
both = pd.concat([nm, res], ignore_index=True)
both = both[both["seed"].isin(res["seed"].unique()) & both["inference"].isin(res["inference"].unique())]
summary = both.groupby(["inference", "relation", "group", "method"])[["AUROC", "AUPRC", "Recall@100"]].mean().reset_index()
summary.to_csv(os.path.join(COV_DIR, "check3x_paper_metrics_all_methods_summary.csv"), index=False)
methods = ["NetMedGPT"] + BASELINES
for grp in ["training edge, missed by walks (nodes never co-occur)", "held-out test edge"]:
    for metric in ["AUPRC", "Recall@100"]:
        print(f"{grp} | {metric}")
        print(summary[summary["group"] == grp].pivot_table(index=["inference", "relation"], columns="method", values=metric)[methods].round(3).to_string())
        print()